# Stage 2: Cryptojacking Classification (Fine-Tuning)

Once you have gathered more cryptojacking binaries and disassembled them into `e:\Thesis\cryptojacking`, run this notebook.

This notebook will:
1. **Balance** the dataset (taking an equal number of benign and malware samples).
2. **Tokenize** using our newly trained Domain-Adapted GraphCodeBERT model.
3. **Add a Classification Head** (`num_labels=2`) on top of the pre-trained weights.
4. **Train** and save the final malware classifier.

In [ ]:
# Ensure you have the required metrics library:
# pip install evaluate scikit-learn

### 1. Data Loading and Balancing

In [ ]:
import os
import glob
import json
import random
from datasets import Dataset

BENIGN_DIR = r'e:\Thesis\benign'
MALWARE_DIR = r'e:\Thesis\cryptojacking'

def load_and_balance_data(benign_dir, malware_dir):
    benign_files = glob.glob(os.path.join(benign_dir, '*.json'))
    malware_files = glob.glob(os.path.join(malware_dir, '*.json'))
    
    print(f"Found {len(benign_files)} benign samples.")
    print(f"Found {len(malware_files)} cryptojacking samples.")
    
    # Balance dataset to the size of the minority class
    min_samples = min(len(benign_files), len(malware_files))
    
    if min_samples == 0:
        raise ValueError("One of the directories is empty! Please collect data first.")
    if min_samples < 50:
        print("\n--- WARNING ---")
        print("You have very few samples. The classifier might overfit easily.")
        print("Consider getting at least 200+ cryptojacking samples before relying on these results.\n")
        
    print(f"Balancing dataset to {min_samples} samples per class...")
    
    random.shuffle(benign_files)
    random.shuffle(malware_files)
    
    selected_benign = benign_files[:min_samples]
    selected_malware = malware_files[:min_samples]
    
    data = []
    
    # Label 0 for Benign
    for f in selected_benign:
        try:
            with open(f, 'r', encoding='utf-8') as file:
                content = json.load(file)
                if 'asm' in content:
                    data.append({'text': content['asm'], 'label': 0})
        except: pass
            
    # Label 1 for Cryptojacking
    for f in selected_malware:
        try:
            with open(f, 'r', encoding='utf-8') as file:
                content = json.load(file)
                if 'asm' in content:
                    data.append({'text': content['asm'], 'label': 1})
        except: pass
        
    random.shuffle(data)
    
    # Split files before tokenizing to prevent data leakage from chunk striding
    split_idx = int(len(data) * 0.8)
    train_data = Dataset.from_list(data[:split_idx])
    eval_data = Dataset.from_list(data[split_idx:])
    
    return train_data, eval_data

train_data, eval_data = load_and_balance_data(BENIGN_DIR, MALWARE_DIR)
print(f"Total balanced dataset size: {len(train_data) + len(eval_data)}")

### 2. Tokenization
We load the tokenizer from our Stage 1 model. Chunking to 512 tokens.

In [ ]:
from transformers import AutoTokenizer

# Point this to your Stage 1 final folder
MODEL_PATH = "./gcb-assembly-final"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

def tokenize_function(examples):
    # Tokenize with overflow to capture the ENTIRE binary assembly in multiple 512 chunks
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        return_overflowing_tokens=True,
        stride=16
    )
    
    # Map the original class label to each newly generated overlapping chunk
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    labels = []
    for mapping_index in sample_mapping:
        labels.append(examples["label"][mapping_index])
    
    tokenized["label"] = labels
    return tokenized

# We drop the old text and label columns, letting the new tokenized ones take over
# Tokenize train and eval separately to prevent overlapping chunks from leaking across splits
train_dataset = train_data.map(tokenize_function, batched=True, remove_columns=["text", "label"])
eval_dataset = eval_data.map(tokenize_function, batched=True, remove_columns=["text", "label"])

### 3. Model Initialization and Metrics
Notice we are using `AutoModelForSequenceClassification` now, not `AutoModelForMaskedLM`.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import torch
import evaluate
import numpy as np

# Load evaluation metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    # Calculate Accuracy 
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    
    # Calculate Precision, Recall, and F1 prioritizing Binary classification for Malware
    # This ensures we don't just default to generic averages, letting us see missed malware
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")["f1"]
    precision = precision_metric.compute(predictions=predictions, references=labels, average="binary")["precision"]
    recall = recall_metric.compute(predictions=predictions, references=labels, average="binary")["recall"]
    
    return {
        "accuracy": acc, 
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# Load Pre-Trained Weights, but with a Classification Head for 2 labels
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, 
    num_labels=2
)

# Note: It is expected to see a warning here about "some weights not being initialized" 
# because the LM Head from Stage 1 is being thrown away and replaced by a randomly initialized Classification Head.

### 4. Training (Fine-tuning the Classifier)

In [ ]:
training_args = TrainingArguments(
    output_dir="./gcb-stage2-classification",
    overwrite_output_dir=False,       # Changed from True to prevent nuking existing checkpoints
    num_train_epochs=10,              # Increased max epochs slightly, relying on EarlyStopping
    per_device_train_batch_size=8,    
    per_device_eval_batch_size=8,
    warmup_ratio=0.06,                # Changed from warmup_steps for consistency with Stage 1
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,               # Keep only the last 2 checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="f1",       # Use F1 score for early stopping/best model instead of loss
    fp16=True,                        
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Stop if F1 plateaus for 3 epochs
)

print("Starting Training...")
trainer.train()

### 5. Final Save

In [ ]:
# Save the final classifier to disk
FINAL_STAGE2_DIR = "./gcb-stage2-final"
trainer.save_model(FINAL_STAGE2_DIR)
tokenizer.save_pretrained(FINAL_STAGE2_DIR)

print(f"Success! Your Malware Classifier is saved at {FINAL_STAGE2_DIR}")